# Inspect DdT notation and facsimiles

This notebook is refreshed from CAMAT by `python scripts/sync_camat_viewer.py`.
Rendering, annotations and watching use the installed CAMAT package. Updates
preserve recognized configuration assignments and back up the previous notebook.
It reads MEI without rewriting it. This viewer is an inspection tool; run
`mei_corpus_checks.ipynb` or `scripts/validate_corpus.py` for general + DdT checks.

Run from this repository, select the CAMAT Python kernel, and configure
`MEI_SOURCE` below. No package installation or tutorial download occurs here.


In [ ]:
from pathlib import Path
import camat
from camat import launch_interactive_facsimile_viewer, read_facsimile_model

CORPUS_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "validation/works.json").is_file()), None)
if CORPUS_ROOT is None:
    raise FileNotFoundError("Start this notebook from the DdT_1_vol_11 repository.")
print("CAMAT:", camat.__version__, camat.__file__)
print("Corpus:", CORPUS_ROOT)


In [ ]:
# Choose a corpus-relative path or another supported CAMAT source.
MEI_SOURCE = str(CORPUS_ROOT / '11_buxtehude_sonatas_final/buxtehude_op1_03_sonata_a_minor_corr.mei')
SHOW_VEROVIO_WARNINGS = False


## 1. Inspect available facsimile records

With `allow_missing_facsimile=True`, `read_facsimile_model` returns a score-only model when no usable facsimile records exist. When records are present, it validates every surface containing measure zones, resolves its graphic, and checks every measure link. Genuinely unresolved `@facs` links still raise because they indicate inconsistent—not merely absent—editorial data.


In [ ]:
facsimile_model = read_facsimile_model(MEI_SOURCE, allow_missing_facsimile=True)
print(f"Viewer mode:   {facsimile_model['viewer_mode']}")
print(f"Measures:      {len(facsimile_model['measures'])}")
if facsimile_model['has_facsimile']:
    print(f"Surfaces:      {len(facsimile_model['surfaces'])}")
    print(f"Linked zones:  {len(facsimile_model['linked'])}")
    print(f"Missing @facs: {len(facsimile_model['missing_facs'])}")
else:
    print(f"Facsimile:     {facsimile_model['facsimile_status']}")
    print("The viewer will render the notation at full width.")


## 2. Configure rendering and reloading

`ALLOW_MISSING_FACSIMILE=True` enables the score-only fallback. In that mode, **Check facsimile** can detect records added later without rerendering unchanged notation. With a linked facsimile, **Reload zones** reuses cached score SVG when only coordinates or `@facs` links changed. **Reload score** also reruns Verovio after notation or layout changes; for HTTP(S) sources it becomes **Reload source** and downloads the MEI again. **Auto-watch MEI** only tracks a local file; it is disabled for HTTP(S) links. `SHOW_ANNOTATIONS` enables the annotation list and score highlights. `ALIGN_TO_FACSIMILE` optionally inserts inferred `<pb>`/`<sb>` markers only into the temporary rendering copy; it never modifies the source MEI. Remote facsimile scans are inlined (`EMBED_REMOTE_GRAPHICS=True`) because the notebook widget iframe cannot fetch third-party IIIF URLs; the launch cell is slower on a first load of a many-surface edition, then cached.


In [ ]:
VEROVIO_INITIAL_PAGE = 1
VEROVIO_ORIENTATION = "portrait"
VEROVIO_BREAKS = "encoded"
VEROVIO_ADJUST_PAGE_HEIGHT = True
ALIGN_TO_FACSIMILE = False

VIEWER_MAX_HEIGHT = 820
FACSIMILE_MAX_WIDTH = 600
ZONE_OPACITY = 0.18

INITIAL_SCORE_ZOOM_PERCENT = 100
INITIAL_FACSIMILE_ZOOM_PERCENT = 100
ZOOM_STEP_PERCENT = 10
MIN_ZOOM_PERCENT = 50
MAX_ZOOM_PERCENT = 300
SHOW_DIAGNOSTIC_TABLE = True
ALLOW_MISSING_FACSIMILE = True
SHOW_ANNOTATIONS = True
ANNOTATION_DISPLAY_LIMIT = 300
# Notebook widget iframes block third-party IIIF <img src>; inline the scans.
EMBED_REMOTE_GRAPHICS = True

AUTO_WATCH_MEI = False
AUTO_WATCH_MODE = "events"
WATCH_INTERVAL_SEC = 2.0
WATCH_SETTLE_SEC = 0.35
WATCH_DEBOUNCE_SEC = 0.4

VEROVIO_OPTIONS = {
    "footer": "none",
    "scale": 35,
    "svgViewBox": True,
    "condense": "none",
}


## 3. Launch the adaptive viewer

Without facsimile records, the notation uses the full viewer width. With linked records, hover or click a measure in either pane to highlight its partner. Changing the Verovio score page automatically selects the facsimile surface referenced by measures on that page. The toolbar has independent −/percentage/+ controls for score and facsimile zoom. **Layout** switches between *Auto* (side by side, stacking on narrow screens such as tablets), *Side by side*, and *Stacked*; drag the divider between the panes to resize them, or double-click it to reset. Both choices are remembered in this browser across reloads. Annotation targets are highlighted separately; click an annotation to navigate to and emphasize its `@plist` or `@tstamp` target.

Hover over any rendered score element to see its element name and source `xml:id` in the **Hovered** row beneath the measure/zone/page information. This works for notes, dynamics, rests, clefs, accidentals, slurs, and other rendered element types, including in score-only mode or measures without facsimile links. The nearest element with a source ID wins: an identified accidental shows its own ID, while a generated notehead or stem resolves to its identified MEI ancestor. Leaving the score or changing pages clears the hovered ID. Nonvisual XML elements have no score area to hover over.


In [ ]:
viewer = launch_interactive_facsimile_viewer(
    MEI_SOURCE,
    verovio_initial_page=VEROVIO_INITIAL_PAGE,
    verovio_orientation=VEROVIO_ORIENTATION,
    verovio_breaks=VEROVIO_BREAKS,
    verovio_adjust_page_height=VEROVIO_ADJUST_PAGE_HEIGHT,
    verovio_options=VEROVIO_OPTIONS,
    viewer_max_height=VIEWER_MAX_HEIGHT,
    facsimile_max_width=FACSIMILE_MAX_WIDTH,
    zone_opacity=ZONE_OPACITY,
    initial_score_zoom_percent=INITIAL_SCORE_ZOOM_PERCENT,
    initial_facsimile_zoom_percent=INITIAL_FACSIMILE_ZOOM_PERCENT,
    zoom_step_percent=ZOOM_STEP_PERCENT,
    min_zoom_percent=MIN_ZOOM_PERCENT,
    max_zoom_percent=MAX_ZOOM_PERCENT,
    show_diagnostic_table=SHOW_DIAGNOSTIC_TABLE,
    show_verovio_warnings=SHOW_VEROVIO_WARNINGS,
    allow_missing_facsimile=ALLOW_MISSING_FACSIMILE,
    show_annotations=SHOW_ANNOTATIONS,
    annotation_display_limit=ANNOTATION_DISPLAY_LIMIT,
    embed_remote_graphics=EMBED_REMOTE_GRAPHICS,
    align_to_facsimile=ALIGN_TO_FACSIMILE,
    auto_watch_mei=AUTO_WATCH_MEI,
    auto_watch_mode=AUTO_WATCH_MODE,
    watch_interval_sec=WATCH_INTERVAL_SEC,
    watch_settle_sec=WATCH_SETTLE_SEC,
    watch_debounce_sec=WATCH_DEBOUNCE_SEC,
)


## What was produced?

- an in-memory inspection model describing either score-only or linked-facsimile mode;
- one or more cached Verovio SVG pages;
- an interactive HTML score view, with a linked facsimile pane when records exist.

No edition data was written. An absent facsimile is allowed; inconsistent records such as unresolved `@facs` links remain visible errors to fix in the editorial workflow. Reload here after editing, then continue to CAMAT's parsing and representation workflow once the page is reviewed.

Verovio preserves `<annot>` elements as empty SVG groups rather than engraving their prose. CAMAT therefore reads annotation text itself: `@plist` points directly to rendered IDs, while `@tstamp` is resolved through Verovio's timemap in the containing measure and filtered by `@staff`/`@layer` when supplied. The diagnostic section also reports facsimile-derived layout suggestions.
